# Configure Python Imports

In order to run our Jupyter Notebook,  we will need the following libraries:
- requests:  This library is used to make HTTP Requests to the TDP API
- json:  Allows us to manipulate files as JSON Objects
- pandas:  This is a very useful library for storing data in tabular structures
- numpy:  Open Source Framework for mathematical computation
- matplotlib:  Library for creating visualizations of your data

In [ ]:
import requests, json, pandas, numpy, matplotlib.pyplot as plt
%matplotlib inline

# Configure Connection Variables

Create and store information on how to connect to the TDP API. Set the following values in the code cell below, inside the empty quotation marks:

* Set `api_root` to point at the correct URL for your TDP environment
 * If your TDP instance URL is: https://platform.tetrascience.com
then your `api_root` should be: `"https://api.tetrascience.com"` - note that you do not need “platform” in this case, this is an exception.
 * If your TDP URL is anything other than “platform.tetrascience.com”, such as: https://platform-eu.tetrascience.com then your `api_root` should be the full hostname with "api" included: `"https://api.platform-eu.tetrascience.com"`
* Set `orgSlug` to be your org slug (ex: `training-jsmith`)
* Set `userToken` to an auth token, either your user's personal JWT or a Service User's token
* To get your user's JWT value, open or return to a Chrome tab with TDP, click on the main hamburger menu in the top left corner and select My Account. You should be presented with the Account details screen. Click the \"Copy Token\" button. Your clipboard will now have the token needed for calling the API.
* Set `userLabel` to match the value that you had entered in Lab 1 for your “user” Label. It should look similar to the following: [first_name]-[last_name]. E.g. if your name is John Smith, you would have used `john-smith`. Verify this value on a file or in the Agent configuration if you do not get results in later steps.


In [ ]:
# Example api_root:
# US Multitenant: "https://api.tetrascience.com" - note that "platform" is not included
# EU Multitenant: "https://api.platform-eu.tetrascience.com"
api_root = ""
orgSlug = ""
userToken = ""
userLabel = ""

In [ ]:
headers = {"x-org-slug": orgSlug, "ts-auth-token": userToken}
SearchURL = api_root + "/v1/datalake/searchEql"

SearchURL

# Set Query to Search for All Empower Projects

The TDP API uses ElasticSearch for indexing the Empower Content.  This powerful tool allows advanced searching against the data to find the appropriate information based upon your use case(s).

In this scenario, we are creating a query to find all of the Empower Data for the Project Names within.  We are using the "aggs" function of the ElasticSearch API to then collect all of the unique Project Names.

In [ ]:
payload = {
    "size": 0,
      "query": {
        "bool": {
          "must": [
            {
              "term": {"idsType": "lcuv-empower"}
            },
            {
              "nested": {
                "path": "labels",
                "query": {
                  "bool": {
                    "must": [
                      {"term": {"labels.name": "user"}},
                      {"term": {"labels.value": userLabel}}
                    ]
                  }
                }
              }
            }
          ]
        }
    },
    "aggs": {
        "unique_project_names": {
           "terms": { "field": "data.project.name", "size": 500 }
        }
    }
}

# Run Search Request and Display Results

We will now use the requests library to make a request to the TDP API.  We have previously configured the connection variables as well as the query we are executing.

Our final line just has the variable "result" in it which tells the Notebook to print the value of the variable.

In [ ]:
request = requests.post(SearchURL, json=payload, headers=headers)

result = request.json()

result

Now let's isolate the results by accessing data by attribute names and assign it to a Pandas DataFrame object for viewing the data as a table.  (We only uploaded one project to our instance and therefore there is only one result. Typically you'd see many results).

We are also renaming the columns in the DataFrame to make them more human readable.

In [ ]:
projects = result.get("aggregations").get("unique_project_names").get("buckets")

df = pandas.DataFrame(projects)
df = df.rename(columns={"key": 'Project Name', "doc_count": "No Injections"})

df

# Now Go Get All Injections for the Project and Display in a Table

First let's create the payload for the API Query.  We are retrieving values from the first row of the previous data table to pass in to the query.  The first value specifies the amount of results we desire.  The second one filters the query to only respond with injections associated with the Project Name in question.

Since this query will return all of the Empower Data Files (Injections) for this specific project, we would like to specify the fields that are relevant for our use case.

The "Fields" variable is being used for specifying the data of interest.  Notice how we use the json_normalize function to allow us to specify data in nested JSON objects (The injection data).  We are also renaming the JSON paths to field names that are more human readable in the DataFrame.

Finally you'll notice that we are using the pandas DataFrame fillna function.  This allows us to replace Null values with something of our choosing.  This will be relevant in the next exercise.

In [ ]:
payload_inj = {
    "size": int(df["No Injections"][0]),
    "query": {
        "bool": {
            "must": [
                {"nested": {
                    "path": "labels",
                    "query": {
                      "bool": {
                        "must": [
                          {"term": {"labels.name": "user"}},
                          {"term": {"labels.value": userLabel}}
                        ]
                      }
                    }
                  }
                },
                {"term": {"data.project.name": df["Project Name"][0]}}
            ]
        }
    }
}

request_inj = requests.post(SearchURL, json=payload_inj, headers=headers)
result_inj = request_inj.json()

allHits = result_inj.get("hits").get("hits")

df_inj_runs = pandas.json_normalize(
    allHits,
    ["_source", ["data", "runs"]],
    record_prefix='_source.data.runs.',
    meta=[["_source", "fileId"]])

df_inj_methods = pandas.json_normalize(
    allHits,
    ["_source", ["data", "methods"]]
    , record_prefix='_source.data.methods.')

df_inj_systems = pandas.json_normalize(
    allHits,
    ["_source", ["data", "systems"]]
    , record_prefix='_source.data.systems.')

df_inj = df_inj_runs.join(df_inj_methods).join(df_inj_systems)

Fields = ["_source.fileId"
          , "_source.data.runs.injection.id"
          , "_source.data.methods.sample_set.name"
          , "_source.data.runs.injection.time.acquisition"
          , "_source.data.runs.injection.volume.value"
          , "_source.data.runs.injection.volume.unit"
         ]

renamedDataFrame = df_inj[Fields].rename(
    columns={"_source.fileId": "File Id"
             , "_source.data.runs.injection.id": "Injection Id"
             , "_source.data.methods.sample_set.name": "Sample Set Name"
             , "_source.data.runs.injection.time.acquisition": "Acquisition Time"
             , "_source.data.runs.injection.volume.value": "Volume"
             , "_source.data.runs.injection.volume.unit": "Unit"})

renamedDataFrame["Sample Set Name"].fillna("Missing Value", inplace=True)

renamedDataFrame

# Display a Bar Chart Showing the Number of Injections per Sample Set

In order to display this bar chart, we will need to isolate the unique Sample Set Names in our data set and assign them to an array.  This also is required to get the Total Number of Injections per Sample Set Name.   Both of these arrays will be used as input for the Bar Chart.

We are using an Open Source Python Framework called MatPlotLib for showing the data visualization.

In [ ]:
sampleSets = renamedDataFrame["Sample Set Name"].unique()
print("Sample Sets As Array:")
print(sampleSets)

sampleSetInjectionCounts = renamedDataFrame.groupby("Sample Set Name").count()["Injection Id"].to_numpy()
print("\nInjection Counts As Array:")
print(sampleSetInjectionCounts)

Now let's configure the Horizontal Bar Chart and plot the chart on the Notebook.

In [ ]:
fig, ax = plt.subplots();

ax.barh(sampleSets, sampleSetInjectionCounts)
ax.set_title("Injections Per Sample Set")
ax.set_xlabel("Injections")

plt.show()

# Sort Data by System Utilization

For our final use case, let's group the Empower Data based upon overall System Run Time. This information can be useful in ensuring your lab personnel balances their runs across all of their equipment which of course can help maximize investment.

We are going to work with the existing data set we have and will query the specific fields of interest.  We are looking for the Injection Run Time (Both the units - Minutes, Hours, etc... and value).  Once we have the DataFrame with the data of interest, we can group the data set by System Name and Run Time Unit.   The new DataFrame can then be displayed and ordered by Run Time Unit and Run Time in a descending manner.

In [ ]:
Fields = ["_source.data.systems.name"
          , "_source.data.runs.injection.run_time.value"
          , "_source.data.runs.injection.run_time.unit"]

renamedDataFrame = df_inj[Fields].rename(
    columns={"_source.data.systems.name": "System Name"
             , "_source.data.runs.injection.run_time.value": "Run Time"
             , "_source.data.runs.injection.run_time.unit": "Run Time Unit"})


renamedDataFrame = renamedDataFrame.groupby(["System Name", "Run Time Unit"]).sum()

renamedDataFrame.sort_values(by=["Run Time Unit", "Run Time"], ascending=False)

# Fetch a complete injection IDS file

In the examples above, data is being fetched from the ElasticSearch index.

Not all of the data is stored in this index, for example `datacubes` data (for the large numeric data set) is excluded from the index.

In cases like this, first we can find a file of interest with ElasticSearch, which gives us a file ID. Then we can use another TDP API endpoint to download the full file including datacubes.

In [ ]:
# First, create a query which will return 1 file from the project of interest
payload_single = {
    # Size 1 to just get 1 file ID
    "size": 1,
    "query": {
        "bool": {
            "must": [
                {"nested": {
                    "path": "labels",
                    "query": {
                      "bool": {
                        "must": [
                          {"term": {"labels.name": "user"}},
                          {"term": {"labels.value": userLabel}}
                        ]
                      }
                    }
                  }
                },
                {"term": {"data.project.name": df["Project Name"][0]}}
            ]
        }
    }
}

# Get the file ID from the result
request_single = requests.post(SearchURL, json=payload_single, headers=headers)
result_single = request_single.json()
file_id = result_single.get("hits").get("hits")[0].get("_source").get("fileId")

print(f"File ID: {file_id}")

# Now fetch the full file using the TDP API
file_url = f"{api_root}/v1/datalake/retrieve"
file_response = requests.get(file_url, headers=headers, params={"fileId": file_id})

# Load the IDS JSON into a dictionary
empower_data = file_response.json()

# Channels, chromatograms and peaks: understanding the Empower IDS data structure

The Empower IDS (Intermediate Data Schema) organizes chromatography data using a relational structure with primary keys (pk) and foreign keys (fk). Next we will look at three of the key tables for chromatogram analysis:

### channels

Contains detector channel information (e.g., PDA Ch1 215nm, System Pressure). Each channel has a unique `pk` (primary key).

**Key fields:**
- `pk`: Primary key (UUID) for the channel
- `name`: Channel name (e.g., "PDA Ch1 215nm@4.8nm")
- `description`: Full description
- `type`: Channel type (e.g., "2D")

### results (with peaks)

These are the results of processing methods created in Empower, such as background subtraction (which leads to a new chromatogram in datacubes) or peak integration (which leads to peak information directly in the results array).

**Key fields:**
- `fk_channel`: Foreign key linking to channels.pk
- `peaks`: Array of detected/integrated peaks with retention times, areas, heights, etc.
- `date_processed`: When the result was processed

### datacubes

Contains the raw chromatogram data of signal against retention time (or against volume in some cases). This links to the channel which produced this data via `fk_channel`.

**Key fields:**
- `fk_channel`: Foreign key linking to channels.pk
- `name`: Same as channel name
- `measures`: Array containing the data values (absorbance, pressure, etc.)
- `dimensions`: Dimension data (e.g. retention time or volume)

Next we'll look at some metadata and data from channels, results (with peaks) and datacubes:

In [ ]:
# Extract channels - the base table with pk
# Path: methods[*].channels[*]
methods = empower_data.get("methods", [])
channels_list = []
for method in methods:
    for channel in method.get("channels", []):
        channels_list.append({
            "channel_pk": channel.get("pk"),
            "channel_name": channel.get("name"),
            "channel_status": channel.get("status")
        })

channels_df = pandas.DataFrame(channels_list)
print(f"Found {len(channels_df)} channels:")
channels_df

In [ ]:
# Extract results with peak counts - linked via fk_channel
results_list = []
for result in empower_data.get("results", []):
    peaks = result.get("peaks", [])
    results_list.append({
        "result_fk_channel": result.get("fk_channel"),
        "result_id": result.get("id"),
        "result_date_processed": result.get("date_processed"),
        "peak_count": len(peaks)
    })

results_df = pandas.DataFrame(results_list)
print(f"Found {len(results_df)} results:")
results_df

In [ ]:
# Extract individual peaks into a DataFrame (one row per peak)
peaks_list = []
for result in empower_data.get("results", []):
    fk_channel = result.get("fk_channel")
    for peak in result.get("peaks", []):
        retention_time = peak.get("retention", {}).get("time", {}).get("value")
        peaks_list.append({
            "peak_fk_channel": fk_channel,
            "peak_number": peak.get("number"),
            "retention_time": retention_time,
            "peak_area": peak.get("area", {}).get("value"),
            "peak_height": peak.get("height", {}).get("value")
        })

peaks_df = pandas.DataFrame(peaks_list)
print(f"Found {len(peaks_df)} peaks:")
peaks_df

In [ ]:
# Extract datacubes - linked via fk_channel
# Includes time_array and signal_array for plotting
datacubes_list = []
for datacube in empower_data.get("datacubes", []):
    measures = datacube.get("measures", [])
    dimensions = datacube.get("dimensions", [])
    
    # Extract time dimension
    time_scale = None
    for dim in dimensions:
        if dim.get("name") == "time":
            time_scale = dim.get("scale", [])
            break
    
    # Get signal values and metadata from measures
    signal_values = []
    measure_unit = None
    if measures:
        measure_unit = measures[0].get("unit")
        values = measures[0].get("value", [])
        if values and len(values) > 0:
            signal_values = values[0] if isinstance(values[0], list) else values
    
    datacubes_list.append({
        "datacube_fk_channel": datacube.get("fk_channel"),
        "datacube_name": datacube.get("name"),
        "datacube_unit": measure_unit,
        "datacube_data_points": len(signal_values),
        "time_array": time_scale,
        "signal_array": signal_values
    })

datacubes_df = pandas.DataFrame(datacubes_list)
print(f"Found {len(datacubes_df)} datacubes:")
# Display summary without the large array columns
datacubes_df[["datacube_fk_channel", "datacube_name", "datacube_unit", "datacube_data_points"]]

In [ ]:
# Left join channels to results and datacubes to create a summary
# This shows which channels have data and peaks available

channel_summary = channels_df.merge(
    results_df,
    left_on="channel_pk",
    right_on="result_fk_channel",
    how="left"
).merge(
    datacubes_df,
    left_on="channel_pk",
    right_on="datacube_fk_channel",
    how="left"
)

# Select relevant columns and add boolean flags
channel_summary["has_results"] = channel_summary["result_fk_channel"].notna()
channel_summary["has_datacube"] = channel_summary["datacube_fk_channel"].notna()

# Create a clean summary view
summary_columns = [
    "channel_name",
    "has_results",
    "peak_count",
    "has_datacube",
    "datacube_unit",
    "datacube_data_points"
]

channel_summary_clean = channel_summary[summary_columns].copy()
channel_summary_clean["peak_count"] = channel_summary_clean["peak_count"].fillna(0).astype(int)
channel_summary_clean["datacube_data_points"] = channel_summary_clean["datacube_data_points"].fillna(0).astype(int)

print("Channel Summary - Data Availability per Channel:")
print("="*60)
channel_summary_clean

# Plot chromatogram data with peaks overlaid

Now we will plot chromatograms with peaks overlaid for each of the channels in this Empower injection data.

Note that 1 injection may have chromatograms from multiple channels, e.g. absorbance detectors at different wavelengths, so we can either select a channel of interest, or plot them all separately.

Also note that the available peak data comes from the processing method run in Empower - all of the settings about peak detection are configured in that processing method. It is also possible to run an injection without processing peaks afterwards, which will lead to a data set containing chromatograms but no peak information.

In [ ]:
# Reuse the DataFrames created earlier: channels_df, peaks_df, datacubes_df
# Join channels with datacubes (inner join - only channels with datacubes)
plot_data = channels_df.merge(
    datacubes_df,
    left_on="channel_pk",
    right_on="datacube_fk_channel",
    how="inner"
)

# Join with peaks (left join - some channels may not have peaks)
# Group peaks by channel first to create a list of peaks per channel
if not peaks_df.empty:
    # Group all peaks for a given channel into an array
    peaks_grouped = peaks_df.groupby("peak_fk_channel").apply(
        lambda x: x.to_dict("records"), include_groups=False
    ).reset_index(name="peaks_list")

    plot_data = plot_data.merge(
        peaks_grouped,
        left_on="channel_pk",
        right_on="peak_fk_channel",
        how="left"
    )
else:
    plot_data["peaks_list"] = None

# Plot each channel's chromatogram with peaks overlaid
for idx, row in plot_data.iterrows():
    channel_name = row["channel_name"]
    time_array = row["time_array"]
    signal_array = row["signal_array"]
    signal_unit = row["datacube_unit"]
    peaks_list = row.get("peaks_list") if pandas.notna(row.get("peaks_list")) else []
    
    if not time_array or not signal_array:
        print(f"Skipping {channel_name}: missing data")
        continue
    
    # Create the figure
    fig, ax = plt.subplots(figsize=(12, 5))
    
    # Plot chromatogram
    ax.plot(time_array, signal_array, 'k-', linewidth=0.8)

    # Overlay peaks
    for peak in peaks_list:
        rt = peak.get("retention_time")
        if rt is not None:
            # Find signal at peak retention time, to vertically offset the peak label
            closest_idx = min(
                range(len(time_array)), 
                key=lambda i: abs(time_array[i] - rt)
            )
            signal_at_peak = signal_array[closest_idx]

            ax.plot(rt, signal_at_peak, 'ko', markersize=4)
            ax.annotate(
                f"{rt:.3f}",
                xy=(rt, signal_at_peak),
                xytext=(0, 10),
                textcoords='offset points',
                ha='center',
                fontsize=8,
                rotation=45,
            )

    ax.set_xlabel("Minutes")
    ax.set_ylabel(signal_unit)
    ax.set_title(channel_name)
    ax.set_xlim(left=0)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print(f"  {len(peaks_list)} peak(s) detected")
    print()